<div align="right"><sub>Notebook 最終更新: 2026-03-25 18:06</sub></div>
<h1><strong>05. AIエージェントの基礎（リトライ版）</strong></h1>

今回からは，LLMを単体で使うのではなく，役割を持った「エージェント」として組み合わせて，複雑なタスクをこなす方法を学びます．
まずは，回答を行う **Executor** と，その回答をチェックする **Critic** の2役を組み合わせた「自己修正ループ」を体験しましょう．

> **モデルの変更**: Qwen3-8B は高い推論能力を持つため，並のタスク（論理パズルなど）は1発で正解を出してしまいます．ここでは，「LLMがやりがちな見落とし（フォーマット違反，NGワード，語尾の制約）」を意図的に設けて，エージェントによる自己修正が働く様子を観察します．

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentence-transformers faiss-cpu peft datasets gradio

import os
import sys
from google.colab import drive

DRIVE_MOUNT_POINT = '/content/drive'
drive.mount(DRIVE_MOUNT_POINT, force_remount=False)

PERSIST_ROOT = os.path.join(DRIVE_MOUNT_POINT, 'MyDrive', 'AIAgent')
PERSIST_INDEX_DIR = os.path.join(PERSIST_ROOT, 'data', 'index')
os.makedirs(PERSIST_INDEX_DIR, exist_ok=True)

REPO_ROOT = '/content/llm_lab'
if not os.path.exists(REPO_ROOT):
    !git clone -b ai_agent https://github.com/akio-kobayashi/llm_lab.git {REPO_ROOT}
else:
    # 最新の修正を適用するための pull
    !cd {REPO_ROOT} && git pull

os.chdir(REPO_ROOT)
src_path = os.path.abspath('src')
if src_path not in sys.path:
    sys.path.append(src_path)

print('現在の作業ディレクトリ:', os.getcwd())
print('永続ディレクトリ:', PERSIST_ROOT)
from src.common import load_llm, generate_text, AGENT_MODEL_ID
from src.agent_core import LLMExecutorCriticAgent, RoleConfig

model, tokenizer = load_llm(model_id=AGENT_MODEL_ID)
print('準備完了')


## **1. チャット関数の準備**


In [ ]:
def llm_chat(system_prompt: str, user_prompt: str, max_tokens: int = 512, temp: float = 0.3):
    return generate_text(model, tokenizer, user_prompt, max_new_tokens=max_tokens, temperature=temp, system_prompt=system_prompt)

---
## **タスク候補 E: 厳密な JSON 出力**
LLMはよく「以下が結果です」と前置きしたり、```json というマークダウンで囲みがちです。Criticがそれを許さず、純粋なJSON文字列のみに修正させます。

In [ ]:
executor_e = RoleConfig(
    name="Executor",
    system_prompt="あなたはJSONデータ生成アシスタントです。ユーザーの要求に従い、JSON形式のみを出力してください。"
)
critic_e = RoleConfig(
    name="Critic",
    system_prompt="あなたは非常に厳格なフォーマットレビュアーです。出力が波括弧 `{` で完全に始まり、`}` で完全に終わっているか、1文字目と最後の文字を厳密にチェックしてください。\nもし ` ```json ` などのマークダウンや、「以下が」などの説明文が1文字でも前後に含まれていれば、必ず抽出漏れとして指摘し、純粋なJSON部分のみに修正するよう求めてください。\n条件を完全に満たす場合のみ「誤りなし」と出力してください。"
)
agent_e = LLMExecutorCriticAgent(llm_chat, role_configs=[executor_e, critic_e])

query_e = "日本の主要な島を4つ挙げ、「islands」というキーを持つ配列としてJSONで出力してください。その他の文章やマークダウンは絶対に含めないでください。"
answer_e, log_e, _ = agent_e.run_pipeline(query_e, max_iterations=2)
print("=== 候補E: 厳密なJSON出力 ===")
print(log_e)
print("\n=== 最終回答 ===")
print(answer_e)

✅ **期待する動作**: Executor が ```json ... ``` を出力してしまう → Critic がそれを発見して「マークダウンが含まれている」と指摘する → Executor が `{` と `}` だけの出力を返す。

---

## **タスク候補 F: NGワードと必須条件の組み合わせ**
LLMは「〜を使うな（否定の指示）」を忘れがちです。また、文末の特定の締めくくりも忘れやすいため、Criticがそこを指摘します。

In [ ]:
executor_f = RoleConfig(
    name="Executor",
    system_prompt="あなたは朝食の提案アシスタントです。必ず指定された条件を忠実に守って回答してください。"
)
critic_f = RoleConfig(
    name="Critic",
    system_prompt="あなたは厳格な校閲者です。以下の3点を1つずつ確認して結果を書き出してください。\n(1) 文章の中に「パン」という単語が含まれていないか（含まれていたら✗）\n(2) 文章の中に「納豆」と「卵」が含まれているか（不足があれば✗）\n(3) 文章の最後が「ごちそうさまでした。」で完全に終わっているか（1文字でも違えば✗）\n1つでも✗があれば「誤りあり」として修正を指示し、すべて条件を満たせば「誤りなし」と出力してください。"
)
agent_f = LLMExecutorCriticAgent(llm_chat, role_configs=[executor_f, critic_f])

query_f = "健康的な朝食について、3文の短い文章で提案してください。ただし、「パン」という言葉は絶対に使わず、「納豆」と「卵」を必ず使ってください。また、一番最後の文は「ごちそうさまでした。」にしてください。"
answer_f, log_f, _ = agent_f.run_pipeline(query_f, max_iterations=2)
print("=== 候補F: NGワードと必須条件 ===")
print(log_f)
print("\n=== 最終回答 ===")
print(answer_f)

✅ **期待する動作**: Executor が「パン」を使ってしまう、あるいは結びの言葉が「〜ごちそうさまでした。」とならずに「〜です。」で終わってしまう → Criticが指摘する。

---

## **タスク候補 G: 語尾の強力な制約**
LLMは長い文章を書く際に、普段の学習データに引っ張られて普通の日本語の語尾に戻ってしまう傾向（ドリフト現象）があります。

In [ ]:
executor_g = RoleConfig(
    name="Executor",
    system_prompt="あなたは猫のアシスタントです。ユーザーの指示に従って文章を書いてください。"
)
critic_g = RoleConfig(
    name="Critic",
    system_prompt="あなたは語尾チェッカーです。回答文を1文ずつ区切り、すべての文の句点（。）の直前が「ニャ」になっているかを厳密に確認してください。例として「〜ですニャ。」「〜だニャ。」はOKですが、「〜です。」「〜ました。」など普通の語尾が1つでも混ざっていれば「誤りあり」として指摘してください。問題なければ「誤りなし」と出力してください。"
)
agent_g = LLMExecutorCriticAgent(llm_chat, role_configs=[executor_g, critic_g])

query_g = "猫の視点で、人間の「お掃除ロボット（ルンバ）」についての感想を、3つの文で語ってください。ただし、すべての文の最後を必ず「ニャ。」にしてください。"
answer_g, log_g, _ = agent_g.run_pipeline(query_g, max_iterations=2)
print("=== 候補G: 語尾の制約 ===")
print(log_g)
print("\n=== 最終回答 ===")
print(answer_g)

✅ **期待する動作**: Executorが「〜お掃除ロボットは怖いですね。でも便利ニャ。」のように、普通の句点を挟んでしまう → Criticがその文を見つけて修正させる。

---

## **結果の確認と本番タスクの決定**
この3つの候補から、モデルが適度につまづき、さらに Critic が正確に誤りを見つけて自己修正ループが綺麗に回るものを本番タスクとして1つ採用します。